In [35]:
import pandas as pd
import numpy as np
from collections import Counter
from sklearn.model_selection import train_test_split
from transformers import (
    RobertaTokenizerFast, 
    RobertaForSequenceClassification,
    get_linear_schedule_with_warmup
)
from sklearn.utils.class_weight import compute_class_weight
import torch
from torch.utils.data import Dataset, DataLoader

SEED=42
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


In [19]:
# Load training data
TRAIN_PATH = './dontpatronizeme_pcl.tsv'

data_train = pd.read_csv(
    TRAIN_PATH,
    sep='\t',
    skiprows=4,
    names=['par_id', 'art_id', 'keyword', 'country_code', 'text', 'label'],
    dtype={'par_id': int, 'art_id': str, 'keyword': str, 'country_code': str, 'text': str, 'label': int}
)

# Convert to binary labels (0-1 = No PCL, 2-4 = PCL)
data_train['binary_label'] = data_train['label'].apply(lambda x: 1 if x >= 2 else 0)
print(f"\nBinary label distribution:")
label_counts = data_train['binary_label'].value_counts()
print(f"  No PCL (0): {label_counts[0]} ({label_counts[0]/len(data_train)*100:.1f}%)")
print(f"  PCL (1): {label_counts[1]} ({label_counts[1]/len(data_train)*100:.1f}%)")


Binary label distribution:
  No PCL (0): 9476 (90.5%)
  PCL (1): 993 (9.5%)


In [20]:
# Create training and validation split from training data
# We'll use 80% for training and 20% for validation (for hyperparameter tuning)
texts = data_train['text'].tolist()
labels = data_train['binary_label'].tolist()

train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts,
    labels,
    test_size=0.2,
    stratify=labels,
    random_state=SEED
)
# 3️⃣ Quick sanity check
print(f"Total examples: {len(texts)}")
print(f"Training examples: {len(train_texts)}")
print(f"Validation examples: {len(val_texts)}")

# Check class distribution

print("\nTraining set class distribution:", Counter(train_labels))
print("Validation set class distribution:", Counter(val_labels))

Total examples: 10469
Training examples: 8375
Validation examples: 2094

Training set class distribution: Counter({0: 7581, 1: 794})
Validation set class distribution: Counter({0: 1895, 1: 199})


In [32]:
# Define PyTorch Dataset class
class PCLDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts  #list of paragraph strings
        self.labels = labels  #list of 0s and 1s
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        
        # Tokenize with max_length=100 (Proposed approach: optimised input representation)
        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
            add_special_tokens=True
        )
        
        return {
            'input_ids': encoding['input_ids'].view(-1), #squeeze()
            'attention_mask': encoding['attention_mask'].view(-1), #squeeze
            'labels': torch.tensor(label, dtype=torch.long)
        }

# Create datasets
print("Creating PyTorch datasets...")
MAX_LENGTH = 128  # Proposed: 128 tokens covers ~95% of dataset???

train_dataset = PCLDataset(train_texts, train_labels, tokenizer, max_length=MAX_LENGTH)
val_dataset = PCLDataset(val_texts, val_labels, tokenizer, max_length=MAX_LENGTH)

print(f"✓ Training dataset created: {len(train_dataset)} samples")
print(f"✓ Validation dataset created: {len(val_dataset)} samples")

# Create dataloaders
BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print(f"\n{'='*50}")
print("DataLoaders created:")
print(f"{'='*50}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Max sequence length: {MAX_LENGTH} tokens (Proposed approach)")

Creating PyTorch datasets...
✓ Training dataset created: 8375 samples
✓ Validation dataset created: 2094 samples

DataLoaders created:
Batch size: 16
Training batches: 524
Validation batches: 131
Max sequence length: 128 tokens (Proposed approach)


In [33]:
# labels = data_train['binary_label'].values
# unique_labels, label_counts = np.unique(labels, return_counts=True)

# total_samples = len(labels)
# num_classes = len(unique_labels)

# class_weights = {}
# for label, count in zip(unique_labels, label_counts):
#     weight = total_samples / (num_classes * count)
#     class_weights[label] = weight

# print(f"{'='*50}")
# print("Class Weights for Loss Function:")
# print(f"{'='*50}")
# for label in sorted(class_weights.keys()):
#     print(f"Class {label} ('{['No PCL', 'PCL'][label]}'): weight = {class_weights[label]:.4f}")

# print(f"\nRatio: Class 1 weight / Class 0 weight = {class_weights[1]/class_weights[0]:.2f}")
# print("✓ This weighting emphasizes minority PCL class during training (Proposed approach)")

Class Weights for Loss Function:
Class 0 ('No PCL'): weight = 0.5524
Class 1 ('PCL'): weight = 5.2714

Ratio: Class 1 weight / Class 0 weight = 9.54
✓ This weighting emphasizes minority PCL class during training (Proposed approach)


In [41]:
# Ensure labels are a 1D numpy array of integers
labels_array = np.array(train_labels, dtype=int)
print(labels_array.shape)

# Compute balanced class weights
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1]), 
    y=labels_array
)

# Convert to PyTorch tensor (for use in CrossEntropyLoss)
class_weights = torch.tensor(class_weights, dtype=torch.float)

print("Class weights:", class_weights)

(8375,)
Class weights: tensor([0.5524, 5.2739])


In [43]:


# Load pre-trained RoBERTa tokenizer and model
MODEL_NAME = 'roberta-base'

print(f"Loading {MODEL_NAME} tokenizer and model...")
tokenizer = RobertaTokenizerFast.from_pretrained(MODEL_NAME)
model = RobertaForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,  # Binary classification
    output_hidden_states=False
)
model.to(device)

print(f"✓ Model loaded successfully")
print(f"  - Model name: {MODEL_NAME}")
print(f"  - Number of labels: 2 (No PCL, PCL)")
print(f"  - Device: {device}")
print(f"  - Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  - Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

Loading roberta-base tokenizer and model...


Loading weights: 100%|█| 197/197 [00:00<00:00, 1082.48it/s, Materializing param=
RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✓ Model loaded successfully
  - Model name: roberta-base
  - Number of labels: 2 (No PCL, PCL)
  - Device: cpu
  - Total parameters: 124,647,170
  - Trainable parameters: 124,647,170


In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import f1_score
from torch.optim import AdamW

LEARNING_RATE = 2e-5

# ✅ Weighted Loss for class imbalance
class_weights = class_weights.to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

# ✅ Optimizer
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)

# ✅ Scheduler
EPOCHS = 3
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

# ✅ Training Loop
best_f1 = 0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    
    for batch in train_loader:
        optimizer.zero_grad()
        
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        
        loss = outputs.loss
        total_loss += loss.item()
        loss.backward()
        optimizer.step()
        scheduler.step()
    
    avg_train_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{EPOCHS} - Average Training Loss: {avg_train_loss:.4f}")
    
    # ✅ Validation
    model.eval()
    val_preds, val_labels = [], []
    
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            predictions = torch.argmax(logits, dim=-1)
            
            val_preds.extend(predictions.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())
    
    f1 = f1_score(val_labels, val_preds)
    print(f"Validation F1 (PCL): {f1:.4f}")
    
    # Save best model
    if f1 > best_f1:
        best_f1 = f1
        model.save_pretrained("./BestModel")
        print("Best model saved!\n")
